# «Космос как инфраструктура» — стартовый framework

Notebook реализует только канонический пересчет и проверки. Метод выбора и управленческую логику создает команда.



In [14]:
# 0. Среда: one-click Colab или локальный clone репозитория.
from pathlib import Path
import sys, json, subprocess, shutil, pandas as pd, numpy as np

REPO_URL = "https://github.com/SpaceEconomyPolicy/test.git"
CLONE_DIR = Path('/content/kep_case')
REQUIRED = [
    Path('case_core.py'),
    Path('data')/'lots.csv',
    Path('data')/'access_modes.csv',
    Path('config')/'case_config.json',
]

def is_case_root(path):
    path = Path(path)
    return all((path / rel).exists() for rel in REQUIRED)

def find_local_root():
    candidates = [Path.cwd(), Path.cwd().parent, CLONE_DIR]
    for candidate in candidates:
        if is_case_root(candidate):
            return candidate.resolve()
    return None

ROOT = find_local_root()
if ROOT is None:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    print('Файлы кейса не найдены локально — клонирую:', REPO_URL)
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(CLONE_DIR)], check=True)
    ROOT = CLONE_DIR.resolve()
    missing = [str(rel) for rel in REQUIRED if not (ROOT / rel).exists()]
    if missing:
        raise FileNotFoundError(
            'Репозиторий загружен, но не хватает обязательных файлов: ' + ', '.join(missing)
        )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from case_core import load_case, evaluate_portfolio, check_constraints
lots, modes, cfg = load_case(ROOT)
print('Case version:', cfg['case_version'], '| root:', ROOT, '| lots:', len(lots), '| modes:', len(modes))
display(lots[['lot_id','territorial_archetype','service','capability_groups','c0_mrub','opex_mrub_per_year','vpub_mrub_per_year','t_rep']])


Case version: 1.1 | root: /content/kep_case | lots: 8 | modes: 3


,lot_id,territorial_archetype,service,capability_groups,c0_mrub,opex_mrub_per_year,vpub_mrub_per_year,t_rep
0,FIRE,Siberian,Siberian forest fire monitoring,EO,320,85,560,0.68
1,FLOOD,FarEast,Flood and landslide monitoring,EO,340,90,600,0.62
2,AGRI,South,Agricultural analytics,EO;PNT/InSAR,260,75,230,0.74
3,INFRA,UralVolga,Infrastructure deformation monitoring,PNT/InSAR,310,80,300,0.71
4,ARCTIC,Arctic,Remote logistics support,SATCOM;PNT/InSAR,460,140,470,0.55
5,TRANS,Central,Transport monitoring,PNT/InSAR;EO,250,70,220,0.77
6,ENV,VolgaCaspian,Environmental monitoring,EO,280,75,360,0.73
7,SSA,Federal,Space situational awareness,SSA,330,95,340,0.64


## 1. Карточка решения команды
Заполните формы. В Colab поля `#@param` отображаются как UI-контролы.

In [15]:
team_name = "" #@param {type:"string"}
decision_method = "Other" #@param ["Weighted MCDA","Pareto/frontier","Rule-based/manual","Optimization","Other"]
strategy_thesis = "" #@param {type:"string"}
print(team_name or 'Команда не указана', '|', decision_method)

VERA | Weighted MCDA


## 2. Опциональный собственный режим доступа
Авторский режим — не способ подобрать коэффициенты под желаемый ответ. Если используете `D`, задайте все коэффициенты, обоснуйте причинную связь и проведите чувствительность. Если `D` считается Public Core, базовый общественно значимый слой должен оставаться бесплатным/недискриминационным.

In [16]:
enable_custom_mode = False #@param {type:"boolean"}
custom_mode_name = "" #@param {type:"string"}
k_c0_D = None #@param {type:"number"}
k_opex_D = None #@param {type:"number"}
k_vpub_D = 0 #@param {type:"number"}
k_anchor_D = None #@param {type:"number"}
k_commercial_D = None #@param {type:"number"}
custom_public_core = False #@param {type:"boolean"}
custom_mode_rationale = "" #@param {type:"string"}

modes_work = modes.copy()
if enable_custom_mode:
    row={'mode_id':'D','name_ru':custom_mode_name,'k_c0':k_c0_D,'k_opex':k_opex_D,'k_vpub':k_vpub_D,'k_anchor':k_anchor_D,'k_commercial':k_commercial_D,'public_core':custom_public_core,'description':custom_mode_rationale}
    modes_work=pd.concat([modes_work,pd.DataFrame([row])],ignore_index=True)
    print('Добавлен режим D. Зафиксируйте rationale и sensitivity в материалах команды.')
else:
    print('Используются только канонические A/B/C.')

Используются только канонические A/B/C.


## 3. Выбор 4 лотов и режимов
Оставьте пустое значение до того, как определитесь. Один лот нельзя выбирать дважды.

In [17]:
lot_1 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_1 = "C" #@param ["A","B","C","D"]
lot_2 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_2 = "C" #@param ["A","B","C","D"]
lot_3 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_3 = "C" #@param ["A","B","C","D"]
lot_4 = "" #@param ["","FIRE","FLOOD","AGRI","INFRA","ARCTIC","TRANS","ENV","SSA"]
mode_4 = "C" #@param ["A","B","C","D"]
selection=[(l,m) for l,m in [(lot_1,mode_1),(lot_2,mode_2),(lot_3,mode_3),(lot_4,mode_4)] if l]
print('Selection:', selection)
if len({l for l,_ in selection}) != len(selection): print('⚠ Один лот выбран более одного раза.')

Selection: [('FIRE', 'A'), ('AGRI', 'C'), ('TRANS', 'C'), ('ENV', 'A')]


## 4. Канонический расчет BASE/STRESS

In [31]:
if len(selection)==4 and len({x[0] for x in selection})==4:
    detail, metrics = evaluate_portfolio(selection, lots, modes_work, cfg)
    display(detail[['lot_id','mode_id','c0_mrub','opex_mrub_per_year','vpub_mrub_per_year','cash_mrub_per_year','t_rep']].round(3))
    display(pd.DataFrame([metrics]).round(3))
    print('BASE')
    display(check_constraints(metrics,cfg,'BASE'))
    print('STRESS')
    display(check_constraints(metrics,cfg,'STRESS'))
else:
    print('Выберите 4 уникальных лота. Расчет будет выполнен после заполнения формы.')

,lot_id,mode_id,c0_mrub,opex_mrub_per_year,vpub_mrub_per_year,cash_mrub_per_year,t_rep
0,FIRE,A,336.0,89.25,560.0,83.75,0.68
1,AGRI,C,254.8,71.25,142.6,127.50,0.74
2,TRANS,C,245.0,66.50,136.4,110.50,0.77
3,ENV,A,294.0,78.75,360.0,76.25,0.73


,selected_lots,c0_mrub,opex_mrub_per_year,vpub_mrub_per_year,cash_mrub_per_year,kcash,t_rep,readiness_1_5,resilience_1_5,scale_1_5,territorial_archetypes,capability_groups,capability_set,public_core_lots
0,4,1129.8,305.75,1199.0,398.0,1.302,0.73,4.525,4.05,4.675,4,2,"[EO, PNT/InSAR]",2


BASE


,constraint,ok
0,exact_lot_count,True
1,territorial_archetypes,True
2,capability_groups,True
3,public_core_lots,True
4,c0_limit,True
5,opex_limit,True
6,vpub_floor,True
7,kcash_floor,True
8,t_rep_floor,True


STRESS


,constraint,ok
0,exact_lot_count,True
1,territorial_archetypes,True
2,capability_groups,True
3,public_core_lots,True
4,c0_limit,True
5,opex_limit,True
6,vpub_floor,True
7,kcash_floor,True
8,t_rep_floor,True


## 5. Собственная decision model (опционально)
Ниже — **стартовая форма**, а не обязательные веса. Если используете MCDA, объясните нормализацию, веса и чувствительность. Если используете Парето/правила/оптимизацию — адаптируйте блок.

In [19]:
w_vpub = None #@param {type:"number"}
w_capex = None #@param {type:"number"}
w_opex = None #@param {type:"number"}
w_kcash = None #@param {type:"number"}
w_trep = None #@param {type:"number"}
w_readiness = None #@param {type:"number"}
w_resilience = None #@param {type:"number"}
w_scale = None #@param {type:"number"}
weights={'vpub':w_vpub,'capex':w_capex,'opex':w_opex,'kcash':w_kcash,'t_rep':w_trep,'readiness':w_readiness,'resilience':w_resilience,'scale':w_scale}
print('Сумма весов:', round(sum(weights.values()),6))
if decision_method=='Weighted MCDA' and abs(sum(weights.values())-1)>1e-9:
    print('⚠ Для weighted MCDA нормализуйте веса или обоснуйте другую схему.')

Сумма весов: 1.0


## 6. Sensitivity hook
Минимум проверьте факторы, которые реально могут изменить ваш управленческий выбор. Пример ниже создает ±20% диапазон для двух выбранных весов, но **не выбирает портфель автоматически**.

In [20]:
sensitivity_weight_1 = "scale" #@param ["vpub","capex","opex","kcash","t_rep","readiness","resilience","scale"]
sensitivity_weight_2 = "scale" #@param ["vpub","capex","opex","kcash","t_rep","readiness","resilience","scale"]
for key in [sensitivity_weight_1,sensitivity_weight_2]:
    base=weights[key]
    print(key, 'range:', round(base*0.8,4), '→', round(base*1.2,4))
print('Добавьте функцию team_score(...) или Парето-анализ в соответствии с вашим методом.')

vpub range: 0.24 → 0.36
kcash range: 0.12 → 0.18
Добавьте функцию team_score(...) или Парето-анализ в соответствии с вашим методом.


## 7. Управленческие поля
Расчет сам по себе не отвечает на вопрос кейса. Пишитие предложения в полях.

In [21]:
payer_opex = "" #@param {type:"string"}
operator_model = "" #@param {type:"string"}
supplier_switch_rule = "" #@param {type:"string"}
replicable_core = "" #@param {type:"string"}
local_adaptation = "" #@param {type:"string"}
stress_decision = "" #@param {type:"string"}
print('Заполнено управленческих полей:', sum(bool(x.strip()) for x in [payer_opex,operator_model,supplier_switch_rule,replicable_core,local_adaptation,stress_decision]), '/ 6')

Заполнено управленческих полей: 6 / 6


## 8. Экспорт результатов
После полного заполнения сохраните воспроизводимые цифры рядом с запиской.

In [32]:
from pathlib import Path
out=Path('results'); out.mkdir(exist_ok=True)
if len(selection)==4 and len({x[0] for x in selection})==4:
    detail, metrics = evaluate_portfolio(selection, lots, modes_work, cfg)
    detail.to_csv(out/'portfolio_detail.csv',index=False)
    with open(out/'portfolio_metrics.json','w',encoding='utf-8') as f: json.dump(metrics,f,ensure_ascii=False,indent=2)
    summary={'team':team_name,'decision_method':decision_method,'strategy_thesis':strategy_thesis,'selection':selection,'weights':weights,'management':{'payer_opex':payer_opex,'operator_model':operator_model,'supplier_switch_rule':supplier_switch_rule,'replicable_core':replicable_core,'local_adaptation':local_adaptation,'stress_decision':stress_decision}}
    with open(out/'team_decision_config.json','w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    print('Сохранено в', out.resolve())
else:
    print('Экспорт появится после выбора 4 уникальных лотов.')

Сохранено в /content/results
